In [171]:
from gymnasium import spaces
import pandas as pd
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import yaml
import random
from copy import deepcopy

In [172]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [173]:
import gc

gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1024**2, "MB")

0.0 MB


In [174]:
num_warmup_episodes = 5_000
batch_size = 256
learning_rate = 1e-3
num_episodes = 200
target_update_steps = 100
gamma = 0.97
alpha = 0.1
epsilon_start = 0.5
epsilon_end = 0.05
epsilon_decay = 0.997
episodes = 1200

In [175]:
class CatchBoxEnv(gym.Env):
    """
    Среда ловли падающих коробок.
    Ширина игрового поля: width (например, 10 единиц).
    Высота игрового поля: height (например, 10 единиц).
    """
    def __init__(self, width=10, height=10):
        super().__init__()
        
        self.width = width
        self.height = height
        
        # 3 действия: 0 - влево, 1 - стоять, 2 - вправо
        self.action_space = spaces.Discrete(3)
        
        # Состояние: [x-координата повозки, x-координата коробки, y-координата коробки]
        # Все значения нормализованы от 0 до 1 для удобства нейросети
        self.observation_space = spaces.Box(
            low=0.0, 
            high=1.0, 
            shape=(3,), 
            dtype=np.float32
        )
        
        self.cart_x = None
        self.box_x = None
        self.box_y = None
        self.score = 0

    def _get_obs(self):
        # Возвращаем координаты, нормализованные от 0.0 до 1.0
        return np.array([
            self.cart_x / (self.width - 1),
            self.box_x / (self.width - 1),
            self.box_y / (self.height - 1)
        ], dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Повозка спавнится по центру внизу
        self.cart_x = self.width // 2
        
        # Первичный спавн коробки наверху
        self._spawn_box()
        self.score = 0
        
        return self._get_obs(), {}

    def _spawn_box(self):
        # Коробка появляется в случайной колонке на самой верхней строчке
        self.box_x = self.np_random.integers(0, self.width)
        self.box_y = self.height - 1

    def step(self, action):
        # 1. Движение повозки
        if action == 0 and self.cart_x > 0:
            self.cart_x -= 1
        elif action == 2 and self.cart_x < self.width - 1:
            self.cart_x += 1
            
        # 2. Падение коробки
        self.box_y -= 1
        
        reward = 0.0
        terminated = False
        truncated = False
        
        # 3. Проверка достигла ли коробка низа (земли)
        if self.box_y == 0:
            # Если x повозки совпадает с x коробки — поймали!
            if self.cart_x == self.box_x:
                reward = 1.0
                self.score += 1
                # Появляется новая коробка сверху
                self._spawn_box()
            else:
                # Промах — игра окончена
                reward = -1.0
                terminated = True
                
        return self._get_obs(), reward, terminated, truncated, {"score": self.score}

    def render(self):
        # Отрисовка текстового поля в консоли
        grid = [["." for _ in range(self.width)] for _ in range(self.height)]
        
        # Отрисовка коробки (B) и повозки (C)
        if 0 <= self.box_y < self.height:
            grid[self.height - 1 - self.box_y][self.box_x] = "📦"
        grid[self.height - 1][self.cart_x] = "🛒"
        
        print("\n".join(["".join(row) for row in grid]))
        print(f"Счет: {self.score}\n" + "="*20)

In [176]:
class Boltzman:
    def __init__(self,actions: list):
        self.actions = actions
    def Generator_actions(self,Q_value, temperature):
        if random.random() < temperature:
            return np.random.choice(self.actions)
        else:
            return np.argmax(Q_value)

In [177]:
class Q_cnn(nn.Module):
    def __init__(self,input_dim: int, out_dim: int,latent_dim: int = 128):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            nn.GELU(),
            nn.Linear(latent_dim, latent_dim//2),
            nn.GELU(),
            nn.Linear(latent_dim//2, out_dim)
        )

    def forward(self, state_tensor: torch.tensor):
        return self.block(state_tensor)

In [178]:
class Buffer():
    def __init__(self,capacity: int = 50000):
        self.capacity = capacity
        self.buffer = []

    def __len__(self):
        return len(self.buffer)

    def add(self,transition:tuple) -> None:
        if len(self.buffer) > self.capacity:
            self.buffer.pop(0)

        self.buffer.append(transition)

    def sample(self, batch_size: int):
            return random.sample(self.buffer, batch_size)


In [179]:
env = CatchBoxEnv()
state = env.reset()
state_actions = [0,1,2]
state_dim = 3

policy = Boltzman(state_actions)
buffer = Buffer()

Q_model = Q_cnn(state_dim,3)
target_Q_model = deepcopy(Q_model)

optimizer = torch.optim.AdamW(Q_model.parameters(),lr=learning_rate)

total_steps = 0
temp = epsilon_start
criterion = nn.SmoothL1Loss()

state

(array([0.5555556, 1.       , 1.       ], dtype=float32), {})

In [180]:
def train_step():
    if len(buffer) < num_warmup_episodes:
        return

    batch = buffer.sample(batch_size)
    states, actions, rewards, next_states, terminateds = zip(*batch)

    states_t = torch.FloatTensor(np.array(states))
    actions_t = torch.LongTensor(actions).unsqueeze(1)
    rewards_t = torch.FloatTensor(rewards)
    next_states_t = torch.FloatTensor(np.array(next_states))
    terminateds_t = torch.FloatTensor(terminateds)
    current_q = Q_model(states_t).gather(1, actions_t).squeeze(1)

    with torch.no_grad():
        next_actions = Q_model(next_states_t).argmax(dim=1, keepdim=True)
        next_q_values = target_Q_model(next_states_t).gather(1, next_actions).squeeze(1)
        targets = rewards_t + gamma * next_q_values * (1 - terminateds_t)

    loss = criterion(current_q,targets)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(Q_model.parameters(), max_norm=1.0)
    optimizer.step()

In [181]:
for epoch in range(1200):
    done = False
    state,_ = env.reset()
    total_reward = 0
    temp = max(temp*epsilon_decay, epsilon_end)

    while not done:
        with torch.no_grad():
            state_t = torch.as_tensor(state,dtype = torch.float32).unsqueeze(0)
            Q_value = Q_model(state_t).squeeze(0).numpy()

        action = policy.Generator_actions(Q_value, temp)
        next_state, reward, terminated,truncated, info = env.step(action)

        done = terminated or truncated
        buffer.add((state, action, reward, next_state, terminated))

        train_step()

        state = next_state
        total_reward += reward
        total_steps += 1
        
        if total_steps % target_update_steps == 0:
            target_Q_model.load_state_dict(Q_model.state_dict())


    if (epoch + 1) % 10 == 0:
        print(f"Эпизод {epoch + 1}. Температура: {temp:.3f}. Награда: {total_reward:.2f}")
        
    
env.close()
    

Эпизод 10. Температура: 0.485. Награда: -1.00
Эпизод 20. Температура: 0.471. Награда: -1.00
Эпизод 30. Температура: 0.457. Награда: -1.00
Эпизод 40. Температура: 0.443. Награда: -1.00
Эпизод 50. Температура: 0.430. Награда: -1.00
Эпизод 60. Температура: 0.418. Награда: -1.00
Эпизод 70. Температура: 0.405. Награда: -1.00
Эпизод 80. Температура: 0.393. Награда: -1.00
Эпизод 90. Температура: 0.382. Награда: -1.00
Эпизод 100. Температура: 0.370. Награда: -1.00
Эпизод 110. Температура: 0.359. Награда: -1.00
Эпизод 120. Температура: 0.349. Награда: -1.00
Эпизод 130. Температура: 0.338. Награда: -1.00
Эпизод 140. Температура: 0.328. Награда: -1.00
Эпизод 150. Температура: 0.319. Награда: -1.00
Эпизод 160. Температура: 0.309. Награда: 1.00
Эпизод 170. Температура: 0.300. Награда: -1.00
Эпизод 180. Температура: 0.291. Награда: -1.00
Эпизод 190. Температура: 0.283. Награда: -1.00
Эпизод 200. Температура: 0.274. Награда: 0.00
Эпизод 210. Температура: 0.266. Награда: -1.00
Эпизод 220. Температура:

In [182]:
env.render()

..........
..........
..........
..........
..........
..........
..........
..........
..........
...🛒.....📦
Счет: 6
